# 03. 모델 평가 — 파인튜닝 8B vs GPT-4o vs o3

## 목적
CON·PEP·RPT 검토 태스크에서 **파인튜닝한 로컬 8B**를 상용 대형 모델(GPT-4o, o3)과 비교한다.
성능·비용·보안 3축으로 평가하여, 소량 도메인 파인튜닝의 실전 가치를 입증한다.

## 핵심 주장
- **성능**: 8B가 GPT-4o/o3에 근접 (특히 '검토' 판정은 우위 가능)
- **비용**: 로컬 8B는 API 비용 0 → 대량 처리 시 격차 극대화
- **보안**: 로컬 실행으로 공공기관 외부 API 금지 요건 충족

## 공정성 원칙
- 세 모델에 **동일한 판정 기준 프롬프트(RULES)** 제공 — 상용 모델이 기준을 몰라 불리해지지 않게
- **동일한 평가셋**(순수 골든, 증강 없음) · **동일한 지표**로 비교
- 평가셋은 학습에 미사용 (test는 최종 1회만)

## 지표
- accuracy, **macro-F1** (클래스 불균형 대응)
- per-class precision/recall/F1 — 특히 **검토(판단보류) recall** (공공 보수성 = 우리 강점)
- 혼동행렬

## 비용 측정
- GPT-4o/o3 호출 시 `response.usage`로 토큰 **실측**
- 1건당 비용 + 대량(1만 건) 시나리오 비용 산출
- o3는 reasoning 토큰도 output으로 과금 → 실측에 반영됨

## 실행 전 채울 자리
1. `JUDGE_PROMPT` — 태스크별 판정 프롬프트 (검증 후 삽입)
2. `OURS_PATH` — 학습된 8B 모델 경로 (학습 후)
3. `predict_ours` 호출 주석 해제 (학습 후)
> GPT-4o·o3 기준선은 프롬프트 확정 후 먼저 실행 가능

## 셀 구성
1. 설정 (API, 모델명)
2. 평가셋 로드 + 태스크 설정
3. 판정 프롬프트 (RULES)
4. 모델별 판정 함수 (+ 토큰 실측)
5. 지표 계산 (순수 파이썬)
6. 실행 (3모델 × 3태스크)
7. 성능·비용 비교표

In [1]:
# gpu 확인
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
import gc, torch
for v in ['base', 'model']:
    if v in dir():
        try: del globals()[v]
        except: pass
if '_ours' in dir() and _ours.get('model') is not None:
    del _ours['model']
gc.collect()
torch.cuda.empty_cache()
print("메모리 정리 완료:", round(torch.cuda.memory_allocated()/1e9,1), "GB 사용중")

메모리 정리 완료: 0.0 GB 사용중


In [3]:
# ======================================================================
# 환경 설정 (OpenAI API 키, GPT-4o/o3 모델명, 우리 모델 경로 OURS_PATH)
# ======================================================================
!pip install -q openai
import json, time
from collections import Counter, defaultdict
from openai import OpenAI
from google.colab import userdata

oai = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
GPT4O_MODEL = "gpt-4o"
O3_MODEL    = "o3"

# 학습된 우리 8B 모델 경로 (학습 후 채움)
OURS_PATH = "/content/drive/Othercomputers/내 노트북/project/Workit/LLM/kanana_finetuned_200_re_best"   # 드라이브에 올린 위치

In [4]:
# bitsandbytes 설치
!pip install -q -U bitsandbytes peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00


In [6]:
# ================
# 모델 파일 확인
# ================
import os
path = "/content/drive/Othercomputers/내 노트북/project/Workit/LLM/kanana_finetuned_200_re_best"   # 로컬 경로
for f in os.listdir(path):
    size = os.path.getsize(os.path.join(path, f)) / 1e6
    print(f"{f}: {size:.0f} MB")

tokenizer_config.json: 0 MB
special_tokens_map.json: 0 MB
README.md: 0 MB
adapter_config.json: 0 MB
training_args.bin: 0 MB


In [7]:
# ====================================================================
# 평가셋 로드 + 태스크 설정 (CON/PEP/RPT test 파일, 라벨, 입력 필드)
# ====================================================================

TASKS = {
    "CON": {
        "file": "CON_test_20.json",
        "gold_field": "label",
        "labels": ["일치","불일치","검토"],
        "input_fields": ["clause_text","refs"],
    },
    "PEP": {
        "file": "PEP_test_20.json",
        "gold_field": "output.label",
        "labels": ["충족","불가","검토"],
        "input_fields": ["input.criteria","input.rfp_excerpt","input.pep_excerpt"],
    },
    "RPT": {
        "file": "RPT_test_20.json",
        "gold_field": "output.label",
        "labels": ["충족","불가","검토"],
        "input_fields": ["input.criteria","input.pep_excerpt","input.rpt_excerpt"],
    },
}

def get_nested(d, path):
    cur = d
    for k in path.split('.'):
        cur = cur.get(k) if isinstance(cur, dict) else None
    return cur

def load_test(task):
    cfg = TASKS[task]
    raw = json.load(open(cfg["file"], encoding="utf-8"))
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    gold = {r["id"]: (get_nested(r, cfg["gold_field"]) if "." in cfg["gold_field"] else r.get(cfg["gold_field"])) for r in data}
    return data, gold, cfg

In [8]:
# =====================================================
# 판정 프롬프트 로드 (추론용 txt 3개를 JUDGE_PROMPT에)
# =====================================================

# 태스크별 판정 프롬프트
def load_prompt(path):
    with open(path, encoding='utf-8') as f:
        return f.read().strip()

JUDGE_PROMPT = {
    "CON": load_prompt("CON_system_prompt_v2.txt"),
    "PEP": load_prompt("PEP_system_prompt_v2.txt"),
    "RPT": load_prompt("RPT_system_prompt_v2.txt"),
}

def build_input(rec, cfg):
    """모델에 줄 입력 (정답 label 제외)"""
    item = {"id": rec["id"]}
    for f in cfg["input_fields"]:
        item[f.split(".")[-1]] = get_nested(rec, f) if "." in f else rec.get(f, "")
    return json.dumps(item, ensure_ascii=False)

FileNotFoundError: [Errno 2] No such file or directory: 'CON_system_prompt_bal.txt'

In [ ]:
# ==================================================================
# 모델별 판정 함수 (GPT-4o/o3 API 호출 + 토큰 실측, predict_ours)
# ==================================================================

# 토큰·비용 누적 (모델별)
USAGE = {"GPT-4o": {"in":0, "out":0, "calls":0},
         "o3":     {"in":0, "out":0, "calls":0}}

def parse_label(txt, labels):
    if not txt: return None
    try:
        p = json.loads(txt)
        for key in ("label","판정","종합판정","재판정"):
            v = p.get(key) if isinstance(p, dict) else None
            if v in labels: return v
    except: pass
    for L in labels:
        if L in txt: return L
    return None

def predict_gpt4o(rec, cfg, task):
    for _ in range(3):
        try:
            r = oai.chat.completions.create(
                model=GPT4O_MODEL,
                messages=[{"role":"system","content":JUDGE_PROMPT[task]},
                          {"role":"user","content":build_input(rec, cfg)}],
                response_format={"type":"json_object"}, temperature=0)
            u = r.usage
            USAGE["GPT-4o"]["in"]  += u.prompt_tokens
            USAGE["GPT-4o"]["out"] += u.completion_tokens
            USAGE["GPT-4o"]["calls"] += 1
            return parse_label(r.choices[0].message.content, cfg["labels"])
        except Exception as e:
            print("  [gpt4o]", e); time.sleep(3)
    return None

def predict_o3(rec, cfg, task):
    for _ in range(3):
        try:
            r = oai.chat.completions.create(
                model=O3_MODEL,
                messages=[{"role":"system","content":JUDGE_PROMPT[task]},
                          {"role":"user","content":build_input(rec, cfg)}],
                response_format={"type":"json_object"}, max_completion_tokens=4000)
            u = r.usage
            USAGE["o3"]["in"]  += u.prompt_tokens
            USAGE["o3"]["out"] += u.completion_tokens   # ← reasoning 토큰 포함됨
            USAGE["o3"]["calls"] += 1
            return parse_label(r.choices[0].message.content, cfg["labels"])
        except Exception as e:
            print("  [o3]", e); time.sleep(3)
    return None


# ── 우리 8B (학습 후 채움) ──
_ours = {"tok": None, "model": None}
def load_ours(path=OURS_PATH):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    _ours["tok"] = AutoTokenizer.from_pretrained(path)
    _ours["model"] = AutoModelForCausalLM.from_pretrained(
        path, device_map="auto", torch_dtype=torch.bfloat16).eval()

def predict_ours(rec, cfg, task):
    import torch
    tok, model = _ours["tok"], _ours["model"]
    msgs = [{"role":"system","content":JUDGE_PROMPT[task]},
            {"role":"user","content":build_input(rec, cfg)}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=512, do_sample=False)
    txt = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_label(txt, cfg["labels"])

In [ ]:
# ========================================
# 우리 8B 모델 로드 (base + LoRA 어댑터)
# ========================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = "kakaocorp/kanana-1.5-8b-instruct-2505"

_ours = {"tok": None, "model": None}

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", dtype=torch.bfloat16
)   # torch_dtype → dtype
_ours["model"] = PeftModel.from_pretrained(base, OURS_PATH).eval()
_ours["tok"]   = AutoTokenizer.from_pretrained(OURS_PATH)

print("우리 모델 로드 완료 (base + 어댑터)")

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# ===========================================================
# 지표 계산 함수 (accuracy, macro-F1, per-class, 혼동행렬)
# ===========================================================

def compute_metrics(gold, pred, labels):
    ids = [i for i in gold if i in pred and pred[i] in labels]
    n = len(ids)
    acc = sum(1 for i in ids if gold[i]==pred[i]) / n if n else 0
    # per-class precision/recall/F1
    per = {}
    f1s = []
    for L in labels:
        tp = sum(1 for i in ids if gold[i]==L and pred[i]==L)
        fp = sum(1 for i in ids if gold[i]!=L and pred[i]==L)
        fn = sum(1 for i in ids if gold[i]==L and pred[i]!=L)
        prec = tp/(tp+fp) if tp+fp else 0
        rec_ = tp/(tp+fn) if tp+fn else 0
        f1 = 2*prec*rec_/(prec+rec_) if prec+rec_ else 0
        per[L] = {"P":prec, "R":rec_, "F1":f1}
        f1s.append(f1)
    macro_f1 = sum(f1s)/len(f1s)
    # 혼동행렬
    conf = {g:{p:0 for p in labels} for g in labels}
    for i in ids: conf[gold[i]][pred[i]] += 1
    return {"n":n, "acc":acc, "macro_f1":macro_f1, "per":per, "conf":conf}

def print_metrics(name, m, labels):
    print(f"\n[{name}] n={m['n']} acc={m['acc']:.3f} macro-F1={m['macro_f1']:.3f}")
    for L in labels:
        p = m["per"][L]
        print(f"  {L}: P={p['P']:.2f} R={p['R']:.2f} F1={p['F1']:.2f}")

In [ ]:
from tqdm.auto import tqdm   # ← 추가
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


if _ours["tok"] is None or _ours["model"] is None:
    print("Ours-8B model and tokenizer are not loaded or were reset. Reloading...")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_44_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE, quantization_config=bnb, device_map="auto", dtype=torch.bfloat16
    )
    _ours["model"] = PeftModel.from_pretrained(base_model, OURS_PATH).eval()
    _ours["tok"]   = AutoTokenizer.from_pretrained(OURS_PATH)
    print("Ours-8B model and tokenizer reloaded successfully.")

ALL = {}
for task in ["CON","PEP","RPT"]:
    data, gold, cfg = load_test(task)
    print(f"\n{'='*60}\n[{task}] test {len(data)}건\n{'='*60}")

    preds = {"GPT-4o": {}, "o3": {}, "Ours-8B": {}}
    # preds = {"Ours-8B": {}}
    for rec in tqdm(data, desc=f"{task} 판정"):   # ← tqdm으로 감쌈
        preds["GPT-4o"][rec["id"]]  = predict_gpt4o(rec, cfg, task)
        preds["o3"][rec["id"]]      = predict_o3(rec, cfg, task)
        preds["Ours-8B"][rec["id"]] = predict_ours(rec, cfg, task)
        time.sleep(1)

    processed_gold = {k: v[0] if isinstance(v, list) else v for k, v in gold.items()}

    ALL[task] = {}
    for name, pred in preds.items():
        if not pred: continue
        m = compute_metrics(processed_gold, pred, cfg["labels"])
        print_metrics(f"{task}-{name}", m, cfg["labels"])
        ALL[task][name] = m

Ours-8B model and tokenizer are not loaded or were reset. Reloading...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Ours-8B model and tokenizer reloaded successfully.

[CON] test 20건


CON 판정:   0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



[CON-Ours-8B] n=20 acc=0.350 macro-F1=0.275
  일치: P=0.30 R=0.43 F1=0.35
  불일치: P=0.40 R=0.57 F1=0.47
  검토: P=0.00 R=0.00 F1=0.00

[PEP] test 20건


PEP 판정:   0%|          | 0/20 [00:00<?, ?it/s]


[PEP-Ours-8B] n=20 acc=0.600 macro-F1=0.444
  충족: P=0.50 R=0.67 F1=0.57
  불가: P=0.67 R=0.89 F1=0.76
  검토: P=0.00 R=0.00 F1=0.00

[RPT] test 21건


RPT 판정:   0%|          | 0/21 [00:00<?, ?it/s]


[RPT-Ours-8B] n=21 acc=0.429 macro-F1=0.308
  충족: P=0.33 R=0.40 F1=0.36
  불가: P=0.47 R=0.70 F1=0.56
  검토: P=0.00 R=0.00 F1=0.00


In [ ]:
# ==========================================================
# 성능·비용 비교표 (단가, 1건당 비용, 대량 처리 시나리오)
# ==========================================================

# ── 단가 (per 1M tokens, USD) — 공식 pricing 확인 후 갱신 ──
PRICE = {
    "GPT-4o": {"in": 2.50, "out": 10.00},
    "o3":     {"in": 2.00, "out":  8.00},   # o3는 reasoning 토큰도 out으로 과금됨
    "Ours-8B":{"in": 0.0,  "out":  0.0},    # 로컬 API 0 (GPU 시간만)
}
USD_KRW = 1400   # 환율 (갱신)

# ── 1. 성능 비교표 ──
print("="*66)
print(f"{'태스크':<6}{'모델':<12}{'acc':<8}{'macro-F1':<10}{'검토R':<8}")
print("-"*66)
for task, models in ALL.items():
    for name, m in models.items():
        rr = m["per"].get("검토",{}).get("R", 0)
        print(f"{task:<6}{name:<12}{m['acc']:<8.3f}{m['macro_f1']:<10.3f}{rr:<8.2f}")

# ── 2. 비용 실측표 ──
print("\n" + "="*66)
print("[비용] 평가셋 전체 실측 (환율 %d원/$ 기준)" % USD_KRW)
print(f"{'모델':<12}{'입력토큰':<10}{'출력토큰':<10}{'건수':<6}{'총$':<10}{'1건당원':<10}")
print("-"*66)
for name in ["GPT-4o", "o3"]:
    u = USAGE[name]
    cost_usd = u["in"]/1e6*PRICE[name]["in"] + u["out"]/1e6*PRICE[name]["out"]
    per_krw = cost_usd*USD_KRW/u["calls"] if u["calls"] else 0
    print(f"{name:<12}{u['in']:<10}{u['out']:<10}{u['calls']:<6}${cost_usd:<9.4f}{per_krw:<10.2f}")
print(f"{'Ours-8B':<12}{'-':<10}{'-':<10}{'-':<6}{'$0':<10}{'~0 (로컬)':<10}")

# ── 3. 대량 처리 시 비용 격차 (예: 계약서 1만 건) ──
print("\n[대량 처리 시나리오] 1만 건 판정 비용")
N = 10000
for name in ["GPT-4o", "o3"]:
    u = USAGE[name]
    if not u["calls"]: continue
    avg_in = u["in"]/u["calls"]; avg_out = u["out"]/u["calls"]
    cost_1 = avg_in/1e6*PRICE[name]["in"] + avg_out/1e6*PRICE[name]["out"]
    print(f"  {name}: ${cost_1*N:,.0f} = 약 {cost_1*N*USD_KRW:,.0f}원")
print(f"  Ours-8B: $0 (로컬 GPU 전기·감가만) — 초기 학습 1회 비용만")

태스크   모델          acc     macro-F1  검토R     
------------------------------------------------------------------
RPT   GPT-4o      0.810   0.785     0.67    
RPT   o3          0.762   0.737     0.50    
RPT   Ours-8B     0.524   0.528     0.50    

[비용] 평가셋 전체 실측 (환율 1400원/$ 기준)
모델          입력토큰      출력토큰      건수    총$        1건당원      
------------------------------------------------------------------
GPT-4o      143350    12330     104   $0.4817   6.48      
o3          141469    84581     103   $0.9596   13.04     
Ours-8B     -         -         -     $0        ~0 (로컬)   

[대량 처리 시나리오] 1만 건 판정 비용
  GPT-4o: $46 = 약 64,841원
  o3: $93 = 약 130,429원
  Ours-8B: $0 (로컬 GPU 전기·감가만) — 초기 학습 1회 비용만


label 분포

In [ ]:
import json
from collections import Counter

# ── 1. 최종 학습 jsonl (실제 학습에 쓴 것) ──
print("="*55)
print("[최종 학습 데이터] kanana_all_train.jsonl")
print("="*55)

rows = [json.loads(l) for l in open('kanana_all_train.jsonl', encoding='utf-8') if l.strip()]
print(f"총 {len(rows)}건\n")

# 태스크별 label 분포
by_task = {}
for r in rows:
    task = r['meta']['task']
    label = r['meta']['label']
    by_task.setdefault(task, Counter())[label] += 1

for task, cnt in by_task.items():
    total = sum(cnt.values())
    print(f"[{task}] 총 {total}건")
    for label, n in cnt.most_common():
        bar = "█" * int(n / total * 30)
        print(f"   {label:<6} {n:>3}건 ({n/total*100:>4.1f}%) {bar}")
    print()

# 전체 합산
all_labels = Counter(r['meta']['label'] for r in rows)
print(f"[전체 합산] {sum(all_labels.values())}건")
for label, n in all_labels.most_common():
    print(f"   {label:<6} {n:>3}건 ({n/sum(all_labels.values())*100:.1f}%)")

[최종 학습 데이터] kanana_all_train.jsonl
총 451건

[CON] 총 150건
   불일치     73건 (48.7%) ██████████████
   일치      48건 (32.0%) █████████
   검토      29건 (19.3%) █████

[PEP] 총 151건
   불가      83건 (55.0%) ████████████████
   충족      51건 (33.8%) ██████████
   검토      17건 (11.3%) ███

[RPT] 총 150건
   불가      62건 (41.3%) ████████████
   충족      62건 (41.3%) ████████████
   검토      26건 (17.3%) █████

[전체 합산] 451건
   불가     145건 (32.2%)
   충족     113건 (25.1%)
   불일치     73건 (16.2%)
   검토      72건 (16.0%)
   일치      48건 (10.6%)


In [ ]:
print("\n" + "="*55)
print("[검증 데이터] kanana_all_valid.jsonl")
print("="*55)

vrows = [json.loads(l) for l in open('kanana_all_valid.jsonl', encoding='utf-8') if l.strip()]
by_task_v = {}
for r in vrows:
    by_task_v.setdefault(r['meta']['task'], Counter())[r['meta']['label']] += 1
for task, cnt in by_task_v.items():
    print(f"[{task}] {dict(cnt)}")


[검증 데이터] kanana_all_valid.jsonl
[CON] {'검토': 3, '일치': 5, '불일치': 7}
[PEP] {'불가': 8, '검토': 2, '충족': 4}
[RPT] {'불가': 6, '충족': 4, '검토': 5}


In [ ]:
import json
from pathlib import Path
from collections import Counter

# =========================
# 파일명 설정
# =========================

CON_PATH = Path("CON_test_20.json")
PEP_PATH = Path("PEP_test_20.json")
RPT_PATH = Path("RPT_test_20.json")

CON_OUT = Path("CON_test_20_v2.json")
PEP_OUT = Path("PEP_test_20_v2.json")
RPT_OUT = Path("RPT_test_20_v2.json")

# RPT_test_03과 RPT_test_04가 중복이라서 04를 삭제할지 여부
REMOVE_RPT_DUPLICATE = True


# =========================
# 공통 함수
# =========================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def get_label(rec):
    """
    CON은 label이 최상위에 있고,
    PEP/RPT는 output.label에 있음.
    RPT는 기존에 ["불가"]처럼 list라서 문자열로 변환.
    """
    if "output" in rec:
        lab = rec["output"].get("label")
    else:
        lab = rec.get("label")

    if isinstance(lab, list):
        return lab[0] if lab else "UNKNOWN"

    return lab


def print_dist(name, data):
    print(f"\n[{name}]")
    print("총 개수:", len(data))
    print("라벨 분포:", dict(Counter(get_label(x) for x in data)))


# =========================
# 1. CON 수정
# - CONT_09: 검토 -> 일치
# - pattern: 검토 -> 일치경계(B)
# =========================

con = load_json(CON_PATH)

for rec in con:
    if rec.get("id") == "CONT_09":
        rec["pattern"] = "일치경계(B)"
        rec["시나리오"] = "보증금률 일치, 반환절차 세부조건"

        rec["사고과정"] = (
            "핵심 조건은 계약보증금률 100분의 10이다. refs 지방계약법 시행령 제51조는 "
            "물품ㆍ용역 등의 계약보증금으로 계약금액의 100분의 10 이상을 내게 해야 한다고 정한다. "
            "clause의 100분의 10은 refs의 핵심 보증금률 기준과 같고, 계약 완료 후 14일 이내 반환은 "
            "반환 절차에 관한 세부조건으로서 현재 refs만으로 불일치를 단정하지 않는다. "
            "따라서 일치경계(B)로 보아 일치."
        )

        rec["label"] = "일치"

        rec["근거"] = (
            "계약보증금을 계약금액의 100분의 10으로 정한 핵심 조건은 지방계약법 시행령 제51조의 "
            "물품ㆍ용역 등 계약보증금 기준과 동일함. 반환기한은 세부 절차로서 현재 refs만으로 "
            "불일치를 단정하지 않음. (참고용 검토이며 법적 판단 아님)"
        )
        break
else:
    raise ValueError("CONT_09를 찾지 못함")

save_json(CON_OUT, con)


# =========================
# 2. PEP 수정
# - PEP-EVAL-013: 검토 -> 충족
# =========================

pep = load_json(PEP_PATH)

for rec in pep:
    if rec.get("id") == "PEP-EVAL-013":
        rec["output"]["label"] = "충족"
        rec["output"]["eval"] = [
            (
                "완전성: 충족 — 제안요청서가 요구한 교육과목, 일정, 대상, 기간, 교육내용, 지원사항이 "
                "과업수행계획서에 모두 반영되어 있음. 특히 클라우드 콘솔 운영 절차는 콘솔 접속과 "
                "자원 조회 안내로, 장애 대응 방법은 장애 발생 시 조치 흐름 안내로 대응되어 "
                "동의어·패러프레이즈 수준의 동일 개념으로 볼 수 있음."
            ),
            (
                "정확성: 충족 — 교육 2회, 회당 4시간, 대상 운영인력 15명이라는 수치가 "
                "제안요청서와 과업수행계획서에서 정확히 일치함"
            )
        ]
        break
else:
    raise ValueError("PEP-EVAL-013을 찾지 못함")

save_json(PEP_OUT, pep)


# =========================
# 3. RPT 수정
# - 모든 output.label: ["불가"] -> "불가" 형태로 변경
# - RPT_test_12: 불가 -> 충족
# - RPT_test_04 중복 삭제 옵션
# =========================

rpt = load_json(RPT_PATH)

# label list -> str 전체 변환
for rec in rpt:
    lab = rec.get("output", {}).get("label")
    if isinstance(lab, list):
        rec["output"]["label"] = lab[0] if lab else "UNKNOWN"

# RPT_test_12 수정
for rec in rpt:
    if rec.get("id") == "RPT_test_12":
        rec["output"]["label"] = "충족"
        rec["output"]["eval"] = [
            (
                "완전성: 충족 — 과업수행계획서에 명시된 착수보고, 요구사항관리대장, 변경관리대장, "
                "개발·테스트, 배포이력서 작성, 형상관리대장 등록, 분기보고, 최종결과보고서 제출 항목이 "
                "사업추진결과보고서의 추진경과, 적용방법론, 개발내용 항목에 모두 대응됨."
            ),
            (
                "검증가능성: 충족 — 착수일자(1월 15일), 접수 주기(매월 1회), 배포 소요기간"
                "(평균 5일 이내), 요구사항 접수 24건, 변경관리대장 작성 12건, 테스트결과서 작성 24건, "
                "배포이력서 작성 9건, 모든 배포이력서의 형상관리대장 등록 완료 등 구체적 수치와 실적이 "
                "제시되어 검증 가능함."
            ),
            (
                "추적성: 충족 — 추진경과의 절차, 적용방법론의 단계별 산출물, 개발내용의 실적 건수가 "
                "서로 연결되어 과업수행계획서의 추진절차와 산출물 흐름에 1:1로 대응됨."
            )
        ]
        break
else:
    raise ValueError("RPT_test_12를 찾지 못함")

# 중복 제거: RPT_test_03과 동일한 RPT_test_04 삭제
if REMOVE_RPT_DUPLICATE:
    rpt = [rec for rec in rpt if rec.get("id") != "RPT_test_04"]

save_json(RPT_OUT, rpt)


# =========================
# 결과 확인
# =========================

print("수정 완료")

print_dist("CON v2", con)
print_dist("PEP v2", pep)
print_dist("RPT v2", rpt)

print("\n저장 파일:")
print(CON_OUT)
print(PEP_OUT)
print(RPT_OUT)

수정 완료

[CON v2]
총 개수: 20
라벨 분포: {'일치': 8, '불일치': 7, '검토': 5}

[PEP v2]
총 개수: 20
라벨 분포: {'충족': 7, '불가': 9, '검토': 4}

[RPT v2]
총 개수: 20
라벨 분포: {'불가': 9, '검토': 5, '충족': 6}

저장 파일:
CON_test_20_v2.json
PEP_test_20_v2.json
RPT_test_20_v2.json
